<a href="https://colab.research.google.com/github/khaled-ibn-elwalid/CNN-ECG-analysis-application/blob/main/training/finalTrainingCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install wfdb

In [ ]:
import os
import zipfile

zip_path = '/content/drive/MyDrive/Master/ptb.zip'  #
dst = '/content/ptb'

os.makedirs(dst, exist_ok=True)

print("Unzipping...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(dst)
print("Done.")

# Check result
print(os.listdir(dst))

In [ ]:
import ast
import os
import warnings
from fractions import Fraction

import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, filtfilt, iirnotch, resample_poly
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

# Used only when ptb_path/scp_statements.csv is absent. Prefer the CSV metadata
# file whenever possible because it is the authoritative PTB-XL label source.
BUILTIN_SCP_SUPERCLASS_MAP = {
    # NORM
    "NORM": "NORM",
    "SR":   "NORM",   # sinus rhythm — normal baseline

    # MI
    "AMI":   "MI", "ALMI":  "MI", "ASMI":  "MI", "ILMI":  "MI",
    "IMI":   "MI", "IPLMI": "MI", "IPMI":  "MI", "LMI":   "MI",
    "PMI":   "MI", "INJAL": "MI", "INJAS": "MI", "INJIL": "MI",
    "INJIN": "MI", "INJLA": "MI", "QWAVE": "MI", "STE_":  "MI",

    # STTC
    "DIG":   "STTC", "ISC_":  "STTC", "ISCAL": "STTC", "ISCAN": "STTC",
    "ISCAS": "STTC", "ISCIL": "STTC", "ISCIN": "STTC", "ISCLA": "STTC",
    "ANEUR": "STTC", "EL":    "STTC", "LNGQT": "STTC", "NDT":   "STTC",
    "NST_":  "STTC", "INVT":  "STTC", "LOWT":  "STTC", "NT_":   "STTC",
    "STD_":  "STTC", "TAB_":  "STTC",

    # CD
    "1AVB":   "CD", "2AVB":   "CD", "3AVB":   "CD",
    "CLBBB":  "CD", "CRBBB":  "CD", "ILBBB":  "CD", "IRBBB":  "CD",
    "IVCD":   "CD", "LAFB":   "CD", "LPFB":   "CD", "WPW":    "CD",
    "ABQRS":  "CD", "AFIB":   "CD", "AFLT":   "CD", "BIGU":   "CD",
    "LPR":    "CD", "PAC":    "CD", "PACE":   "CD", "PRC(S)": "CD",
    "PSVT":   "CD", "PVC":    "CD", "SARRH":  "CD", "SBRAD":  "CD",
    "STACH":  "CD", "SVARR":  "CD", "SVTAC":  "CD", "TRIGU":  "CD",

    # HYP
    "LAO/LAE": "HYP", "LVH":   "HYP", "RAO/RAE": "HYP",
    "RVH":     "HYP", "SEHYP": "HYP", "HVOLT":   "HYP",
    "LVOLT":   "HYP", "VCLVH": "HYP",
}


SCALERS = {
    "zscore": StandardScaler,
    "minmax": MinMaxScaler,
    "robust": RobustScaler,
}

In [ ]:
def bandpass_filter(signal, fs, lowcut=0.5, highcut=40.0, order=4):
    nyquist = fs / 2.0
    highcut = min(highcut, nyquist * 0.99)

    if lowcut <= 0 or lowcut >= highcut:
        raise ValueError(
            f"Invalid bandpass range for fs={fs}: lowcut={lowcut}, highcut={highcut}"
        )

    b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype="band")
    return filtfilt(b, a, signal, axis=0)


def notch_filter(signal, fs, freq=50.0, quality=30.0):
    if freq >= fs / 2.0:
        raise ValueError("Notch frequency must be below Nyquist")

    b, a = iirnotch(freq / (fs / 2.0), quality)
    return filtfilt(b, a, signal, axis=0)


def resample_signal(signal, orig_fs, target_fs=100):
    signal = signal.astype(np.float64, copy=False)
    if np.isclose(orig_fs, target_fs):
        return signal

    ratio = Fraction(float(target_fs) / float(orig_fs)).limit_denominator(1000)
    return resample_poly(signal, ratio.numerator, ratio.denominator, axis=0)


def normalize(signal, method="zscore", clip=None):
    if signal.ndim != 2:
        raise ValueError(f"Expected 2D signal, got {signal.shape}")
    if method not in SCALERS:
        raise ValueError(f"Unknown normalization method: {method}")

    norm = SCALERS[method]().fit_transform(signal.astype(np.float64))
    norm = norm.astype(np.float32)

    if clip is not None:
        if len(clip) != 2 or clip[0] >= clip[1]:
            raise ValueError(f"clip must be (low, high), got {clip}")
        norm = np.clip(norm, clip[0], clip[1])

    return norm


def segment_windows(signal, fs, window_sec=10):
    """Return non-overlapping fixed windows and drop the last incomplete one."""
    win = int(round(window_sec * fs))
    if win <= 0:
        raise ValueError(f"window_sec * fs must be positive, got {window_sec} * {fs}")

    n_windows = signal.shape[0] // win
    trimmed = signal[: n_windows * win]
    return trimmed.reshape(n_windows, win, signal.shape[1])

In [ ]:
DB_CSV        = os.path.join(dst, 'ptbxl_database.csv')
SCP_CSV       = os.path.join(dst, 'scp_statements.csv')

df = pd.read_csv(DB_CSV, index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(ast.literal_eval)

# ── Load SCP statements and keep only diagnostic ones ─────────────────────
scp = pd.read_csv(SCP_CSV, index_col=0)
scp_diag = scp[scp.diagnostic == 1].copy()

print(f'Total records : {len(df):,}')
print(f'Diagnostic SCP codes: {len(scp_diag)}')
df.head(3)

In [ ]:
def load_scp_superclass_map(ptb_path, classes=CLASSES, allow_builtin_fallback=True):
    """
    Load PTB-XL SCP-code to diagnostic-superclass mapping.

    Expected authoritative file:
        ptb_path/scp_statements.csv

    Your current folder does not include this file, so the built-in fallback is
    used unless you add scp_statements.csv beside ptbxl_database.csv.
    """
    scp_path = os.path.join(ptb_path, "scp_statements.csv")

    if os.path.exists(scp_path):
        scp = pd.read_csv(scp_path, index_col=0)
        if "diagnostic_class" not in scp.columns:
            raise ValueError("scp_statements.csv is missing 'diagnostic_class'")

        if "diagnostic" in scp.columns:
            scp = scp[scp["diagnostic"].astype(bool)]

        scp = scp.dropna(subset=["diagnostic_class"])
        class_set = set(classes)
        return {
            str(code): str(row["diagnostic_class"])
            for code, row in scp.iterrows()
            if str(row["diagnostic_class"]) in class_set
        }

    if not allow_builtin_fallback:
        raise FileNotFoundError(f"Missing PTB-XL label metadata: {scp_path}")

    warnings.warn(
        "scp_statements.csv not found. Using built-in PTB-XL superclass mapping. "
        "For publication-quality experiments, add scp_statements.csv to ptb_path.",
        RuntimeWarning,
    )
    return dict(BUILTIN_SCP_SUPERCLASS_MAP)


def scp_to_multihot(scp_str, scp_superclass_map, classes=CLASSES, threshold=0.0):
    """
    Convert PTB-XL scp_codes into a multi-hot vector over:
        NORM, MI, STTC, CD, HYP
    """
    try:
        scp_dict = ast.literal_eval(str(scp_str))
    except (ValueError, SyntaxError):
        return np.zeros(len(classes), dtype=np.float32)

    y = np.zeros(len(classes), dtype=np.float32)
    class_to_index = {class_name: i for i, class_name in enumerate(classes)}

    for code, likelihood in scp_dict.items():
        try:
            likelihood = float(likelihood)
        except (TypeError, ValueError):
            continue
        if likelihood <= threshold:
            continue

        superclass = code if code in class_to_index else scp_superclass_map.get(code)
        if superclass in class_to_index:
            y[class_to_index[superclass]] = 1.0

    return y


def expand_labels_per_segment(record_labels, n_segments):
    if record_labels.ndim != 1:
        raise ValueError(
            f"record_labels must be 1D (K,), got {record_labels.shape}. "
            "Pass one record's label vector at a time."
        )
    return np.repeat(record_labels[np.newaxis, :], n_segments, axis=0)

In [ ]:
def preprocess_record(
    record_path,
    target_fs=100,
    window_sec=10,
    norm_method="zscore",
    clip_range=(-5, 5),
    notch_freq=50.0,
):
    try:
        record = wfdb.rdrecord(record_path)
        signal = record.p_signal
        fs = record.fs

        if signal is None or np.all(np.isnan(signal)):
            return None

        if signal.ndim != 2 or signal.shape[1] != 12:
            n_leads = None if signal.ndim != 2 else signal.shape[1]
            warnings.warn(f"Skipping {record_path}: expected 12 leads, got {n_leads}")
            return None

        if np.any(np.isnan(signal)):
            signal = (
                pd.DataFrame(signal)
                .interpolate(method="linear", limit_direction="both")
                .values
            )

        signal = bandpass_filter(signal, fs)

        if notch_freq is not None and notch_freq < fs / 2.0:
            signal = notch_filter(signal, fs, freq=notch_freq)

        signal = resample_signal(signal, fs, target_fs)
        signal = normalize(signal, method=norm_method, clip=clip_range)
        return segment_windows(signal, target_fs, window_sec)

    except Exception as exc:
        warnings.warn(f"Failed: {record_path} - {exc}")
        return None

In [ ]:
def build_dataset(
    ptb_path,
    target_fs=100,
    window_sec=10,
    norm_method="zscore",
    clip_range=(-5, 5),
    record_column="filename_lr",
    label_threshold=0.0,
):
    csv_path = os.path.join(ptb_path, "ptbxl_database.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"ptbxl_database.csv not found in: {ptb_path}")

    df = pd.read_csv(csv_path)
    required_columns = {record_column, "scp_codes", "strat_fold"}
    missing = sorted(required_columns.difference(df.columns))
    if missing:
        raise ValueError(f"ptbxl_database.csv missing columns: {missing}")

    scp_superclass_map = load_scp_superclass_map(ptb_path)

    X_all, y_all, fold_all = [], [], []
    skipped = 0
    iterator = df.iterrows()
    if tqdm is not None:
        iterator = tqdm(iterator, total=len(df), desc="Loading records")

    for _, row in iterator:
        record_path = os.path.join(ptb_path, str(row[record_column]))
        if not os.path.exists(record_path + ".dat") or not os.path.exists(
            record_path + ".hea"
        ):
            warnings.warn(f"Missing .dat or .hea for: {record_path}")
            skipped += 1
            continue

        label_vec = scp_to_multihot(
            row["scp_codes"],
            scp_superclass_map=scp_superclass_map,
            threshold=label_threshold,
        )
        if label_vec.sum() == 0:
            skipped += 1
            continue

        segs = preprocess_record(
            record_path=record_path,
            target_fs=target_fs,
            window_sec=window_sec,
            norm_method=norm_method,
            clip_range=clip_range,
        )
        if segs is None or len(segs) == 0:
            skipped += 1
            continue

        n_segments = len(segs)
        X_all.append(segs)
        y_all.append(expand_labels_per_segment(label_vec, n_segments))
        fold_all.extend([row["strat_fold"]] * n_segments)

    if not X_all:
        raise RuntimeError(
            "No records were loaded. Check ptb_path, record_column, files, and labels."
        )

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    y = np.concatenate(y_all, axis=0).astype(np.float32)
    folds = np.asarray(fold_all, dtype=np.int64)

    print(f"\nDone - total records: {len(df)} | skipped: {skipped}")
    print(f"X shape  : {X.shape}")
    print(f"y shape  : {y.shape}")
    for i, class_name in enumerate(CLASSES):
        print(f"  {class_name:6s}: {int(y[:, i].sum())} positive segments")

    return X, y, folds


def split_by_official_folds(X, y, folds):
    train_mask = folds <= 8
    val_mask = folds == 9
    test_mask = folds == 10

    return {
        "train": (X[train_mask], y[train_mask]),
        "val": (X[val_mask], y[val_mask]),
        "test": (X[test_mask], y[test_mask]),
    }

In [ ]:
def build_record_table(df, ptb_path, classes=CLASSES, threshold=0.0):
    required_columns = {"ecg_id", "patient_id", "strat_fold", "scp_codes"}
    missing = sorted(required_columns.difference(df.columns))
    if missing:
        raise ValueError(f"ptbxl_database.csv missing columns: {missing}")

    scp_superclass_map = load_scp_superclass_map(ptb_path)
    labels = np.stack(
        [
            scp_to_multihot(scp, scp_superclass_map, classes, threshold)
            for scp in df["scp_codes"]
        ]
    )

    table = pd.DataFrame(
        {
            "record_id": df["ecg_id"],
            "patient_id": df["patient_id"],
            "strat_fold": df["strat_fold"],
        }
    )

    valid = labels.sum(axis=1) > 0
    n_drop = int((~valid).sum())
    if n_drop > 0:
        warnings.warn(f"{n_drop} records had no matching class and were dropped.")

    table = table[valid].reset_index(drop=True)
    labels = labels[valid]

    print(f"Records loaded : {len(table)}")
    print(
        f"Class counts   : "
        f"{ {class_name: int(labels[:, i].sum()) for i, class_name in enumerate(classes)} }"
    )

    return table, labels


def patient_wise_split(
    table,
    labels,
    use_official_folds=True,
    test_size=0.2,
    val_size=0.1,
    seed=42,
):
    if use_official_folds:
        train_mask = table["strat_fold"].isin(range(1, 9))
        val_mask = table["strat_fold"] == 9
        test_mask = table["strat_fold"] == 10
    else:
        patients = table["patient_id"].unique()
        train_pat, test_pat = train_test_split(
            patients, test_size=test_size, random_state=seed
        )
        adjusted_val = val_size / (1.0 - test_size)
        train_pat, val_pat = train_test_split(
            train_pat, test_size=adjusted_val, random_state=seed
        )
        train_mask = table["patient_id"].isin(train_pat)
        val_mask = table["patient_id"].isin(val_pat)
        test_mask = table["patient_id"].isin(test_pat)

    splits = {
        "train": (table[train_mask].reset_index(drop=True), labels[train_mask.values]),
        "val": (table[val_mask].reset_index(drop=True), labels[val_mask.values]),
        "test": (table[test_mask].reset_index(drop=True), labels[test_mask.values]),
    }

    for name, (split_table, _) in splits.items():
        n_patients = len(split_table["patient_id"].unique())
        print(f"{name:5s}: {len(split_table):5d} records | {n_patients:5d} patients")

    return splits


def filter_unlabeled(labels, table):
    valid = labels.sum(axis=1) > 0
    n_dropped = int((~valid).sum())
    if n_dropped > 0:
        warnings.warn(f"{n_dropped} records had no matching class - dropped.")
    return table[valid].reset_index(drop=True), labels[valid]

In [ ]:
from tqdm.auto import tqdm
dst = '/content/ptb'

X, y, folds = build_dataset(dst, record_column="filename_lr")
splits = split_by_official_folds(X, y, folds)
print(
        "Split shapes:",
        splits["train"][0].shape,
        splits["val"][0].shape,
        splits["test"][0].shape,
    )

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

# ── Config ─────────────────────────────────────────────────────────────────
BATCH_SIZE = 64
NUM_WORKERS = 2          # set 0 if you get DataLoader errors in Colab
PIN_MEMORY = True        # speeds up CPU→GPU transfer
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Dataset ────────────────────────────────────────────────────────────────
class ECGDataset(Dataset):
    """
    Wraps numpy (N, 1000, 12) signals and (N, 5) labels.

    - Permutes to (12, 1000) so Conv1d sees (channels, length).
    - Optionally applies light augmentations (train only).
    """

    def __init__(self, X: np.ndarray, y: np.ndarray, augment: bool = False):
        # store as float32 tensors on CPU; GPU transfer happens in the loop
        self.X = torch.tensor(X, dtype=torch.float32)   # (N, 1000, 12)
        self.y = torch.tensor(y, dtype=torch.float32)   # (N, 5)
        self.augment = augment

    # ------------------------------------------------------------------
    def __len__(self) -> int:
        return len(self.X)

    # ------------------------------------------------------------------
    def __getitem__(self, idx: int):
        x = self.X[idx]          # (1000, 12)
        y = self.y[idx]          # (5,)

        if self.augment:
            x = self._augment(x)

        x = x.permute(1, 0)     # → (12, 1000)  ← Conv1d expects (C, L)
        return x, y

    # ------------------------------------------------------------------
    @staticmethod
    def _augment(x: torch.Tensor) -> torch.Tensor:
        """
        Light augmentations applied per-segment at training time.
        x shape: (1000, 12)
        """
        # 1. Gaussian noise
        if torch.rand(1) < 0.5:
            x = x + torch.randn_like(x) * 0.01

        # 2. Amplitude scaling
        if torch.rand(1) < 0.5:
            scale = 0.9 + torch.rand(1) * 0.2   # uniform in [0.9, 1.1]
            x = x * scale

        # 3. Random time-axis flip (polarity inversion)
        if torch.rand(1) < 0.5:
            x = torch.flip(x, dims=[0])

        # 4. Baseline wander (add a slow sine wave per lead)
        if torch.rand(1) < 0.3:
            t = torch.linspace(0, 2 * torch.pi, x.shape[0])
            freq = 0.5 + torch.rand(1) * 1.5    # 0.5–2 Hz
            amp  = torch.rand(1) * 0.05         # up to 5% amplitude
            wander = (amp * torch.sin(freq * t)).unsqueeze(1)  # (1000, 1)
            x = x + wander

        return x


# ── Build DataLoaders ───────────────────────────────────────────────────────
def make_loaders(splits: dict, batch_size: int = BATCH_SIZE):
    """
    splits = output of split_by_official_folds(X, y, folds)
    Returns train_loader, val_loader, test_loader
    """
    X_tr, y_tr = splits["train"]
    X_va, y_va = splits["val"]
    X_te, y_te = splits["test"]

    train_ds = ECGDataset(X_tr, y_tr, augment=True)
    val_ds   = ECGDataset(X_va, y_va, augment=False)
    test_ds  = ECGDataset(X_te, y_te, augment=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=True,          # avoids a tiny last batch messing up BatchNorm
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size * 2,   # no grads → can fit 2× batch
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    return train_loader, val_loader, test_loader


# ── Instantiate ─────────────────────────────────────────────────────────────
train_loader, val_loader, test_loader = make_loaders(splits)

# ── Quick sanity check ──────────────────────────────────────────────────────
xb, yb = next(iter(train_loader))
print(f"Batch x : {xb.shape}   dtype={xb.dtype}")   # → (64, 12, 1000)
print(f"Batch y : {yb.shape}   dtype={yb.dtype}")   # → (64, 5)
print(f"x range : [{xb.min():.3f}, {xb.max():.3f}]")
print(f"Device  : {DEVICE}")
print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

In [ ]:
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================================================
# Residual Block
# =========================================================

class ResBlock1D(nn.Module):
    """
    Two-layer Residual Block for 1D signals.

    Main path:
        Conv -> BN -> LeakyReLU -> Dropout -> Conv -> BN

    Skip path:
        Identity OR 1x1 Conv -> BN (if shape changes)

    Output:
        LeakyReLU(main + skip)
    """

    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        kernel: int = 7,
        stride: int = 1,
        dropout: float = 0.1,
    ):
        super().__init__()

        pad = kernel // 2

        self.main = nn.Sequential(
            nn.Conv1d(
                in_ch,
                out_ch,
                kernel_size=kernel,
                stride=stride,
                padding=pad,
                bias=False,
            ),
            nn.BatchNorm1d(out_ch),

            nn.LeakyReLU(
                negative_slope=0.01,
                inplace=True
            ),

            nn.Dropout(p=dropout),

            nn.Conv1d(
                out_ch,
                out_ch,
                kernel_size=kernel,
                stride=1,
                padding=pad,
                bias=False,
            ),

            nn.BatchNorm1d(out_ch),
        )

        # Projection shortcut if dimensions change
        if stride != 1 or in_ch != out_ch:

            self.skip = nn.Sequential(
                nn.Conv1d(
                    in_ch,
                    out_ch,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),

                nn.BatchNorm1d(out_ch),
            )

        else:
            self.skip = nn.Identity()

        self.act = nn.LeakyReLU(
            negative_slope=0.01,
            inplace=True
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        residual = self.skip(x)
        out = self.main(x)

        return self.act(out + residual)


# =========================================================
# Attention Pooling
# =========================================================

class AttentionPooling(nn.Module):
    """
    Learns temporal attention weights and performs
    weighted temporal pooling.

    Input:
        (B, C, L)

    Output:
        (B, C)
    """

    def __init__(self, in_ch: int):
        super().__init__()

        self.score = nn.Sequential(

            nn.Conv1d(in_ch, in_ch // 2, kernel_size=1),

            nn.Tanh(),

            nn.Conv1d(in_ch // 2, 1, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # Attention weights
        w = torch.softmax(self.score(x), dim=-1)

        # Weighted temporal pooling
        return (x * w).sum(dim=-1)


# =========================================================
# Full ECG CNN
# =========================================================

class ECG1DCNN(nn.Module):
    """
    Architecture:

    Stem:
        Conv(K=15) -> BN -> LeakyReLU -> MaxPool/2

    Stage 0:
        2 × ResBlock(64 -> 64, stride=1)

    Stage 1:
        2 × ResBlock(64 -> 128, first block stride=2)

    Stage 2:
        2 × ResBlock(128 -> 256, first block stride=2)

    Head:
        AttentionPooling -> Dropout -> Linear
    """

    def __init__(
        self,
        n_leads: int = 12,
        n_classes: int = 5,
        base_ch: int = 64,
        blocks_per_stage: tuple = (2, 2, 2),
        kernel: int = 7,
        dropout: float = 0.3,
    ):
        super().__init__()

        # =================================================
        # Stem
        # =================================================

        self.stem = nn.Sequential(

            nn.Conv1d(
                n_leads,
                base_ch,
                kernel_size=15,
                stride=1,
                padding=7,
                bias=False,
            ),

            nn.BatchNorm1d(base_ch),

            nn.LeakyReLU(
                negative_slope=0.01,
                inplace=True
            ),

            nn.MaxPool1d(
                kernel_size=2,
                stride=2,
            ),
        )

        # =================================================
        # Residual stages
        # =================================================

        channel_schedule = [
            base_ch * (2 ** i)
            for i in range(len(blocks_per_stage))
        ]

        stages: list[nn.Module] = []

        in_ch = base_ch

        for stage_idx, n_blocks in enumerate(blocks_per_stage):

            out_ch = channel_schedule[stage_idx]

            for block_idx in range(n_blocks):

                # Downsample only on first block
                # of stages 1+
                stride = (
                    2
                    if (block_idx == 0 and stage_idx > 0)
                    else 1
                )

                stages.append(

                    ResBlock1D(
                        in_ch=in_ch,
                        out_ch=out_ch,
                        kernel=kernel,
                        stride=stride,
                        dropout=dropout,
                    )
                )

                in_ch = out_ch

        self.stages = nn.Sequential(*stages)

        # =================================================
        # Classification head
        # =================================================

        self.gap = AttentionPooling(in_ch)

        self.dropout = nn.Dropout(p=dropout)

        self.fc = nn.Linear(in_ch, n_classes)

        # Initialize weights
        self._init_weights()

    # =====================================================
    # Weight initialization
    # =====================================================

    def _init_weights(self) -> None:

        for m in self.modules():

            if isinstance(m, nn.Conv1d):

                nn.init.kaiming_normal_(
                    m.weight,
                    mode="fan_out",
                    nonlinearity="leaky_relu",
                )

            elif isinstance(m, nn.BatchNorm1d):

                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

            elif isinstance(m, nn.Linear):

                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    # =====================================================
    # Forward pass
    # =====================================================

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x:
                Tensor of shape (B, 12, 1000)

        Returns:
            logits:
                Tensor of shape (B, n_classes)
        """

        x = self.stem(x)       # (B, 64, 500)

        x = self.stages(x)     # (B, 256, 125)

        x = self.gap(x)        # (B, 256)

        x = self.dropout(x)

        logits = self.fc(x)    # (B, n_classes)

        return logits


# =========================================================
# Quick test
# =========================================================

if __name__ == "__main__":

    model = ECG1DCNN().to(DEVICE)

    x = torch.randn(8, 12, 1000).to(DEVICE)

    y = model(x)

    print("Input shape :", x.shape)
    print("Output shape:", y.shape)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
)

# =========================================================
# Device
# =========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

# =========================================================
# Model
# =========================================================

model = ECG1DCNN(
    n_leads=12,
    n_classes=5,
).to(DEVICE)

# =========================================================
# Loss Function
# =========================================================

"""
PTB-XL is usually MULTI-LABEL classification.

That means:
One ECG can contain multiple diagnoses.

So we use:
BCEWithLogitsLoss

IMPORTANT:
- expects RAW logits
- internally applies sigmoid

FIX 3: pos_weight added to handle class imbalance.
Compute from your training labels:
    n_neg / n_pos  per class

Example for PTB-XL superclasses (approximate):
    NORM ~28%  → weight ~2.6
    MI   ~20%  → weight ~4.0
    STTC ~23%  → weight ~3.3
    CD   ~20%  → weight ~4.0
    HYP  ~ 9%  → weight ~10.1

Replace these values with ones computed from your actual dataset.
"""



def compute_pos_weights(loader, n_classes=5):
    all_labels = []

    for _, labels in loader:
        all_labels.append(labels.numpy())          # labels shape: (batch, 5)

    all_labels = np.concatenate(all_labels, axis=0)  # FIX: was np.stack → (N, 5)

    n_pos = all_labels.sum(axis=0)
    n_neg = len(all_labels) - n_pos
    weights = n_neg / (n_pos + 1e-8)

    print("Computed pos_weights:")
    for name, w, p, n in zip(
        ["NORM", "MI", "STTC", "CD", "HYP"],
        weights.tolist(),    # FIX: convert to plain Python floats first
        n_pos.tolist(),
        n_neg.tolist(),
    ):
        print(f"  {name:<6} | pos={int(p):>5} | neg={int(n):>5} | weight={w:.2f}")

    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


pos_weight = compute_pos_weights(train_loader)
pos_weight[4] *= 1.5
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# =========================================================
# Optimizer
# =========================================================

"""
AdamW:
- stable
- modern default
- good for CNNs
- handles weight decay properly
"""

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

# =========================================================
# Scheduler
# =========================================================

"""
OneCycleLR changes learning rate dynamically.

Learning rate:
- increases
- then decreases

Benefits:
- faster convergence
- better generalization
"""

EPOCHS = 60

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer=optimizer,
    max_lr=1e-3,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
)

# =========================================================
# Mixed Precision
# =========================================================

"""
Mixed precision:
- faster GPU training
- lower VRAM usage

autocast:
    uses float16 automatically when safe

GradScaler:
    prevents numerical instability
"""

scaler = torch.amp.GradScaler('cuda')

# =========================================================
# TRAINING FUNCTION
# =========================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scheduler,
    scaler,
    device,
):
    """
    Trains model for ONE epoch.

    Returns:
        average training loss
    """

    # Put model into training mode
    model.train()

    running_loss = 0.0

    # tqdm = progress bar
    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False,
    )

    for signals, labels in progress_bar:

        # ---------------------------------------------
        # Move data to GPU/CPU
        # ---------------------------------------------

        signals = signals.to(device)
        labels = labels.to(device).float()

        # ---------------------------------------------
        # Reset previous gradients
        # ---------------------------------------------

        optimizer.zero_grad()

        # ---------------------------------------------
        # Mixed precision forward pass
        # ---------------------------------------------

        with torch.amp.autocast('cuda'):

            # Forward pass
            logits = model(signals)

            # Compute loss
            loss = criterion(logits, labels)

        # ---------------------------------------------
        # Backpropagation
        # ---------------------------------------------

        scaler.scale(loss).backward()

        # ---------------------------------------------
        # FIX 1: Unscale gradients then clip
        # Prevents exploding gradients from silently
        # corrupting weights during FP16 training.
        # ---------------------------------------------

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # ---------------------------------------------
        # Update weights
        # ---------------------------------------------

        scaler.step(optimizer)

        # Update scaler
        scaler.update()

        # Update learning rate scheduler
        scheduler.step()

        # ---------------------------------------------
        # Track loss
        # ---------------------------------------------

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=loss.item()
        )

    # Average loss across batches
    epoch_loss = running_loss / len(loader)

    return epoch_loss


# =========================================================
# EVALUATION FUNCTION
# =========================================================


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    threshold=0.5,
):
    model.eval()

    running_loss = 0.0

    all_labels = []
    all_probs = []

    for signals, labels in tqdm(
        loader,
        desc="Evaluating",
        leave=False,
    ):
        signals = signals.to(device)
        labels = labels.to(device).float()

        with torch.cuda.amp.autocast():
            logits = model(signals)
            loss = criterion(logits, labels)

        running_loss += loss.item()

        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().float())
        all_labels.append(labels.cpu().float())

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    preds = (all_probs >= threshold).astype(int)

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0,
    )

    try:
        auroc = roc_auc_score(
            all_labels,
            all_probs,
            average="macro",
        )
    except ValueError:
        auroc = 0.0

    avg_loss = running_loss / len(loader)

    return avg_loss, macro_f1, auroc, all_labels, all_probs


# =========================================================
# MAIN TRAINING LOOP
# =========================================================

best_f1 = 0.0

PATIENCE = 7
patience_counter = 0

for epoch in range(EPOCHS):

    print(f"\n========== Epoch {epoch+1}/{EPOCHS} ==========")

    # =================================================
    # TRAINING
    # =================================================

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=DEVICE,
    )

    # =================================================
    # VALIDATION
    # =================================================

    val_loss, val_f1, val_auc, _, _ =evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
    )

    # =================================================
    # PRINT RESULTS
    # =================================================

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val F1     : {val_f1:.4f}")
    print(f"Val AUROC  : {val_auc:.4f}")

    # =================================================
    # SAVE BEST MODEL
    # =================================================

    if val_f1 > best_f1:

        best_f1 = val_f1

        torch.save(model.state_dict(), "best_ecg_model_weights.pt")

        print("Best model saved!")

        patience_counter = 0

    else:
        patience_counter += 1

    # =================================================
    # EARLY STOPPING
    # =================================================

    if patience_counter >= PATIENCE:

        print("\nEarly stopping triggered.")
        break


# =========================================================
# FINAL TEST EVALUATION
# =========================================================



# FIX 4: map_location=DEVICE ensures the checkpoint loads
# correctly regardless of where it was saved (GPU vs CPU).

print("\nLoading best model...")
model = ECG1DCNN(
    n_leads=12,
    n_classes=5,
).to(DEVICE)

model.load_state_dict(
    torch.load("best_ecg_model_weights.pt", map_location=DEVICE, weights_only=True)
)

test_loss, test_f1, test_auc, _, _ = evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
)

print("\n========== FINAL TEST RESULTS ==========")

print(f"Test Loss  : {test_loss:.4f}")
print(f"Test F1    : {test_f1:.4f}")
print(f"Test AUROC : {test_auc:.4f}")


In [ ]:
print(best_f1)
print(patience_counter)

In [ ]:
def find_youden_threshold(y_true, y_probs):
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)

    # roc_curve appends a threshold > max(y_probs), skip it
    thresholds = np.clip(thresholds, 0, 1)

    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    return thresholds[best_idx], j_scores[best_idx]

In [ ]:
from sklearn.metrics import roc_curve, f1_score, roc_auc_score
val_loss, val_f1, val_auc, all_labels, all_probs = evaluate(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=DEVICE,
)

# Find optimal threshold per class
CLASS_NAMES = ["NORM", "MI", "STTC", "CD", "HYP"]
class_thresholds = []

for i in range(all_labels.shape[1]):
    thresh, j_val = find_youden_threshold(all_labels[:, i], all_probs[:, i])
    class_thresholds.append(thresh)
    print(f"{CLASS_NAMES[i]}: Threshold = {thresh:.4f}, J = {j_val:.4f}")

In [ ]:
model.load_state_dict(torch.load("best_ecg_model_weights.pt", map_location=DEVICE))

# Get test probabilities
test_loss, test_f1, test_auc, test_labels, test_probs = evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
)

# Apply per-class Youden thresholds
preds = np.stack([
    (test_probs[:, i] >= class_thresholds[i]).astype(int)
    for i in range(len(class_thresholds))
], axis=1)

# Final metrics
final_f1 = f1_score(test_labels, preds, average="macro", zero_division=0)
per_class_f1 = f1_score(test_labels, preds, average=None, zero_division=0)

print(f"\nTest F1 (Youden thresholds): {final_f1:.4f}")
print(f"Test AUROC: {test_auc:.4f}")
print("\nPer-class F1:")
for name, f1 in zip(["NORM", "MI", "STTC", "CD", "HYP"], per_class_f1):
    print(f"  {name}: {f1:.4f}")

In [ ]:
# 1. Confirm loaders are the ones being used
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# 2. Confirm augmentation is ON for train, OFF for val/test
print(f"Train augment: {train_loader.dataset.augment}")   # should be True
print(f"Val augment:   {val_loader.dataset.augment}")     # should be False
print(f"Test augment:  {test_loader.dataset.augment}")    # should be False

# 3. Confirm batch sizes
print(f"Train batch size: {train_loader.batch_size}")     # should be 64
print(f"Val batch size:   {val_loader.batch_size}")       # should be 128
print(f"Test batch size:  {test_loader.batch_size}")      # should be 128

# 4. Confirm shape is correct for the model (channels first)
xb, yb = next(iter(train_loader))
print(f"x shape: {xb.shape}")   # must be (64, 12, 1000) not (64, 1000, 12)
print(f"y shape: {yb.shape}")   # must be (64, 5)

In [ ]:
# Save everything you need for inference
torch.save({
    "model_state_dict": model.state_dict(),
    "class_thresholds": class_thresholds,
    "class_names": ["NORM", "MI", "STTC", "CD", "HYP"],
}, "ecg_final.pt")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# =========================================================
# 1. DEFINE THE OPTIMIZATION FUNCTION
# =========================================================

def get_optimal_thresholds(val_labels, val_probs):
    """Finds the threshold (0.01 to 0.99) that maximizes F1 for each class."""
    num_classes = val_labels.shape[1]
    best_thresholds = []

    candidate_thresholds = np.linspace(0.01, 0.99, 99)

    for i in range(num_classes):
        y_true = val_labels[:, i]
        y_probs_class = val_probs[:, i]

        best_f1 = 0.0
        best_thresh = 0.5

        for thresh in candidate_thresholds:
            y_pred = (y_probs_class >= thresh).astype(int)
            current_f1 = f1_score(y_true, y_pred, zero_division=0)

            if current_f1 > best_f1:
                best_f1 = current_f1
                best_thresh = thresh

        best_thresholds.append(best_thresh)

    return best_thresholds

# =========================================================
# 2. FIND THRESHOLDS ON VALIDATION SET
# =========================================================

print("--- Step 1: Finding Optimal Thresholds (Validation Set) ---")

# Run evaluation on Validation loader
_, _, _, val_labels, val_probs = evaluate(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=DEVICE,
)

class_names = ["NORM", "MI", "STTC", "CD", "HYP"]
optimal_thresholds = get_optimal_thresholds(val_labels, val_probs)

for i, name in enumerate(class_names):
    print(f"  {name:<6} | Optimal Thresh: {optimal_thresholds[i]:.2f}")

# =========================================================
# 3. APPLY TO TEST SET AND COMPARE
# =========================================================

print("\n--- Step 2: Testing Optimized Thresholds (Test Set) ---")

# Run evaluation on Test loader
# (We only need the raw probabilities and true labels)
_, test_f1_default, _, test_labels, test_probs = evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
)

# Convert our list of optimal thresholds to a numpy array for easy math
thresh_array = np.array(optimal_thresholds)

# Apply the custom thresholds across all predictions simultaneously
optimized_test_preds = (test_probs >= thresh_array).astype(int)

# Calculate the new overall Macro F1
optimized_test_f1 = f1_score(
    test_labels,
    optimized_test_preds,
    average="macro",
    zero_division=0
)

print(f"Original Test F1 (0.5 threshold) : {test_f1_default:.4f}")
print(f"Optimized Test F1 (Custom)       : {optimized_test_f1:.4f}")

print("\nNew Per-Class F1 Scores (Test Set):")
for i, name in enumerate(class_names):
    class_f1 = f1_score(test_labels[:, i], optimized_test_preds[:, i], zero_division=0)
    print(f"  {name:<6}: {class_f1:.4f}")